STEP 1: duplicate row detection

In [1]:
import pandas as pd

df = pd.read_csv(
    "All_Tourism_Reviews.csv")



In [2]:
duplicates = df.duplicated()

print("Total rows:", len(df))
print("Exact duplicates:", duplicates.sum())

Total rows: 22344
Exact duplicates: 1


In [3]:
duplicate_rows = df[df.duplicated(keep=False)]

print(duplicate_rows)

        City         Place  Page                             Review_Text  \
16081  Paris  Eiffel Tower    33  You've seen one, you've seen them all.   
16082  Paris  Eiffel Tower    33  You've seen one, you've seen them all.   

              Review_Date Reviewer_Location  Rating  
16081  September 26, 2020             Idaho       2  
16082  September 26, 2020             Idaho       2  


In [4]:
df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 22343


STEP 2: Convert Review_Date to datetime


In [5]:

df["Review_Date"] = pd.to_datetime(
    df["Review_Date"],
    errors="coerce"
)

print(df["Review_Date"].head())

0   2019-02-17
1   2019-02-17
2   2019-02-17
3   2019-02-16
4   2019-02-16
Name: Review_Date, dtype: datetime64[ns]


STEP 3A: Check review distribution by year


In [6]:
print(
    df["Review_Date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Review_Date
2018       8
2019    4717
2020    4022
2021    2043
2022    3557
2023    3754
2024    4000
2025     233
2026       9
Name: count, dtype: int64


STEP 3B: Keep only reviews from 2019 to 2024


In [7]:

df = df[
    (df["Review_Date"].dt.year >= 2019) &
    (df["Review_Date"].dt.year <= 2024)
].copy()

df.reset_index(drop=True, inplace=True)

print("Reviews remaining:", len(df))

Reviews remaining: 22093


In [8]:
# verification

print("Earliest date:", df["Review_Date"].min())
print("Latest date:", df["Review_Date"].max())

print("\nReviews per year:")
print(
    df["Review_Date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Earliest date: 2019-01-03 00:00:00
Latest date: 2024-12-31 00:00:00

Reviews per year:
Review_Date
2019    4717
2020    4022
2021    2043
2022    3557
2023    3754
2024    4000
Name: count, dtype: int64


STEP 4: Create temporal columns year, month, month_year


In [9]:

df["Year"] = df["Review_Date"].dt.year

df["Month"] = df["Review_Date"].dt.month

df["Month_Year"] = df["Review_Date"].dt.to_period("M").astype(str)

df.head()

,City,Place,Page,Review_Text,Review_Date,Reviewer_Location,Rating,Year,Month,Month_Year
0,Agra,Taj Mahal,0,An amazing experience,2019-02-17,NaN,5,2019,2,2019-02
1,Agra,Taj Mahal,0,Awesome place,2019-02-17,"Dharmendra Pahadiya Pune, India",5,2019,2,2019-02
2,Agra,Taj Mahal,0,Definitely a wonder of the world,2019-02-17,NaN,5,2019,2,2019-02
3,Agra,Taj Mahal,0,Mesmerizing,2019-02-16,NaN,5,2019,2,2019-02
4,Agra,Taj Mahal,0,A place really called as wonder,2019-02-16,NaN,5,2019,2,2019-02


### Step 5: Rating Validation.

In [10]:

# Convert Rating to numeric
df["Rating"] = pd.to_numeric(
    df["Rating"],
    errors="coerce"
)

# Check rating distribution
print("Rating distribution:")
print(
    df["Rating"]
    .value_counts(dropna=False)
    .sort_index()
)

# Find missing or invalid ratings
invalid_ratings = df[
    df["Rating"].isna()
    |
    (~df["Rating"].between(1, 5))
]

print("\nTotal reviews:", len(df))
print("Invalid / missing ratings:", len(invalid_ratings))

# Show problematic rows
display(
    invalid_ratings[
        [
            "City",
            "Place",
            "Review_Text",
            "Review_Date",
            "Rating"
        ]
    ].head(20)
)

# Create a flag instead of immediately deleting reviews
df["Rating_Valid"] = df["Rating"].between(1, 5)

print("\nValid ratings:", df["Rating_Valid"].sum())
print("Invalid ratings:", (~df["Rating_Valid"]).sum())

Rating distribution:
Rating
1     1046
2      854
3     2176
4     4927
5    13090
Name: count, dtype: int64

Total reviews: 22093
Invalid / missing ratings: 0


,City,Place,Review_Text,Review_Date,Rating



Valid ratings: 22093
Invalid ratings: 0


### Step 6 — Missing-value analysis

#### Step 6A: Missing value analysis

In [11]:
missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
})

display(missing_report)

,Missing_Count,Missing_Percent
City,0,0.0
Place,0,0.0
Page,0,0.0
Review_Text,0,0.0
Review_Date,0,0.0
Reviewer_Location,3358,15.2
Rating,0,0.0
Year,0,0.0
Month,0,0.0
Month_Year,0,0.0


#### Step 6B: Handle Reviewer Location

In [12]:
df["Reviewer_Location"] = (
    df["Reviewer_Location"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

df.loc[
    df["Reviewer_Location"] == "",
    "Reviewer_Location"
] = "Unknown"

print(
    df["Reviewer_Location"]
    .value_counts()
    .head(10)
)

Reviewer_Location
Unknown                        3358
London, United Kingdom          646
Dubai, United Arab Emirates     419
New York City, New York         207
1 contribution                  205
2 contributions                 197
3 contributions                 179
Miami, Florida                  152
Singapore, Singapore            147
Paris, France                   131
Name: count, dtype: int64


### Step 7 — Prepare review text correctly

#### Step 7A: Text Cleaning Functions

In [13]:
import re

def clean_for_model(text):
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def clean_for_keywords(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

#### Step 7B: Apply Text Cleaning

In [14]:
df["Review_Text"] = (
    df["Review_Text"]
    .fillna("")
    .astype(str)
)

df["Model_Text"] = (
    df["Review_Text"]
    .apply(clean_for_model)
)

df["Clean_Text"] = (
    df["Review_Text"]
    .apply(clean_for_keywords)
)

display(
    df[
        [
            "Review_Text",
            "Model_Text",
            "Clean_Text"
        ]
    ].head()
)

,Review_Text,Model_Text,Clean_Text
0,An amazing experience,An amazing experience,an amazing experience
1,Awesome place,Awesome place,awesome place
2,Definitely a wonder of the world,Definitely a wonder of the world,definitely a wonder of the world
3,Mesmerizing,Mesmerizing,mesmerizing
4,A place really called as wonder,A place really called as wonder,a place really called as wonder


### Step 8 — Remove only unusable reviews

In [15]:
empty_review_mask = (
    df["Model_Text"]
    .str.strip()
    .eq("")
)

print(
    "Empty reviews:",
    empty_review_mask.sum()
)

Empty reviews: 1


In [16]:
df = (
    df[
        ~empty_review_mask
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Reviews remaining:",
    len(df)
)

Reviews remaining: 22092


### Step 9 — Normalize city and attraction names

#### Step 9A: Inspect

In [17]:
print(
    sorted(
        df["City"]
        .dropna()
        .unique()
    )
)

print(
    sorted(
        df["Place"]
        .dropna()
        .unique()
    )
)

['Agra', 'Bali', 'Barcelona', 'Cairo', 'Dubai', 'Marrakech', 'Miami', 'Paris', 'Rio de Janeiro', 'Rome']
['Agra Fort', 'Arc de Triomphe', 'Bahia Palace', 'Bayside Marketplace', 'Burj Khalifa', 'Casa Batlló', 'Colosseum', 'Copacabana Beach', 'Coptic Cairo', 'Dubai Frame', 'Dubai Mall', 'Dubai Miracle Garden', 'Egyptian Museum', 'Eiffel Tower', 'Escadaria Selarón', 'Gothic Quarter', 'Ipanema Beach', 'Jardin Majorelle', 'Jemaa el-Fnaa', 'Khan el-Khalili Bazaar', 'La Rambla', 'Le Jardin Secret', 'Lempuyang Temple', 'Louvre Museum', 'Mehtab Bagh', 'Miami Beach Boardwalk', 'Miami Seaquarium', 'Montmartre', 'Mosque of Muhammad Ali', 'Mount Batur', 'Notre-Dame Cathedral', 'Pantheon', 'Park Güell', 'Saadian Tombs', 'Sacred Monkey Forest Sanctuary', 'Sagrada Familia', 'Santuário Cristo Redentor', 'Sugarloaf Mountain', 'Taj Mahal', 'Tegallalang Rice Terrace', 'The Dubai Fountain', 'The Pyramids Of Giza', 'Trevi Fountain', 'Uluwatu Temple', 'Vatican Museums', 'Vizcaya Museum & Gardens', 'Wildlife 

#### Step 9B: Basic Cleanup

In [18]:
print(
    sorted(
        df["City"]
        .dropna()
        .unique()
    )
)

print(
    sorted(
        df["Place"]
        .dropna()
        .unique()
    )
)

['Agra', 'Bali', 'Barcelona', 'Cairo', 'Dubai', 'Marrakech', 'Miami', 'Paris', 'Rio de Janeiro', 'Rome']
['Agra Fort', 'Arc de Triomphe', 'Bahia Palace', 'Bayside Marketplace', 'Burj Khalifa', 'Casa Batlló', 'Colosseum', 'Copacabana Beach', 'Coptic Cairo', 'Dubai Frame', 'Dubai Mall', 'Dubai Miracle Garden', 'Egyptian Museum', 'Eiffel Tower', 'Escadaria Selarón', 'Gothic Quarter', 'Ipanema Beach', 'Jardin Majorelle', 'Jemaa el-Fnaa', 'Khan el-Khalili Bazaar', 'La Rambla', 'Le Jardin Secret', 'Lempuyang Temple', 'Louvre Museum', 'Mehtab Bagh', 'Miami Beach Boardwalk', 'Miami Seaquarium', 'Montmartre', 'Mosque of Muhammad Ali', 'Mount Batur', 'Notre-Dame Cathedral', 'Pantheon', 'Park Güell', 'Saadian Tombs', 'Sacred Monkey Forest Sanctuary', 'Sagrada Familia', 'Santuário Cristo Redentor', 'Sugarloaf Mountain', 'Taj Mahal', 'Tegallalang Rice Terrace', 'The Dubai Fountain', 'The Pyramids Of Giza', 'Trevi Fountain', 'Uluwatu Temple', 'Vatican Museums', 'Vizcaya Museum & Gardens', 'Wildlife 

In [20]:
place_mapping = {
    "Park Guell": "Park Güell",
    "Park_Guell": "Park Güell"
}

df["Place"] = df["Place"].replace(place_mapping)

### Step 10 — Language detection

In [21]:
%pip install langdetect -q
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42

def detect_language(text):
    if not isinstance(text, str):
        return "unknown"

    text = text.strip()

    if len(text) < 20:
        return "unknown"

    try:
        return detect(text)

    except:
        return "unknown"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
df["Language"] = (
    df["Model_Text"]
    .apply(detect_language)
)

print(
    df["Language"]
    .value_counts()
)

Language
en         11296
unknown     9281
fr           327
ro           307
de           113
af           109
it            83
es            73
nl            73
no            72
ca            71
id            64
da            42
tl            32
pt            31
et            25
cy            15
fi            14
sv            13
pl             9
so             9
hu             6
hr             6
sw             6
vi             4
tr             4
sq             2
lt             2
sk             1
cs             1
sl             1
Name: count, dtype: int64


In [23]:
df["Is_English"] = (
    df["Language"] == "en"
)

english_percentage = (
    df["Is_English"].mean()
    * 100
)

print(
    f"English reviews: {english_percentage:.2f}%"
)

English reviews: 51.13%


#### Step 10B: Language Distribution

In [24]:
city_language = pd.crosstab(
    df["City"],
    df["Language"]
)

display(city_language)

Language,af,ca,cs,cy,da,de,en,es,et,fi,...,sk,sl,so,sq,sv,sw,tl,tr,unknown,vi
City,,,,,,,,,,,,,,,,,,,,,
Agra,7,6,0,0,2,12,815,3,2,1,...,0,0,1,1,0,1,3,0,433,2
Bali,20,3,1,2,1,9,1031,9,1,3,...,0,0,1,0,1,1,2,0,911,0
Barcelona,16,10,0,3,4,9,1635,16,4,1,...,0,0,2,0,1,0,4,0,917,0
Cairo,2,10,0,0,5,5,843,3,3,2,...,0,0,0,0,0,0,3,1,815,0
Dubai,15,14,0,0,5,14,1595,7,2,0,...,0,0,2,1,1,2,7,0,1176,1
Marrakech,9,10,0,1,7,18,1350,5,4,2,...,0,0,1,0,3,0,5,0,874,0
Miami,13,7,0,4,8,11,1228,6,2,1,...,1,0,2,0,3,2,3,3,819,0
Paris,12,5,0,2,7,9,1017,5,2,2,...,0,1,0,0,4,0,1,0,1209,0
Rio de Janeiro,6,2,0,3,2,10,723,12,1,1,...,0,0,0,0,0,0,3,0,917,0


In [25]:
city_language_pct = pd.crosstab(
    df["City"],
    df["Language"],
    normalize="index"
) * 100

display(city_language_pct.round(2))

Language,af,ca,cs,cy,da,de,en,es,et,fi,...,sk,sl,so,sq,sv,sw,tl,tr,unknown,vi
City,,,,,,,,,,,,,,,,,,,,,
Agra,0.52,0.45,0.00,0.00,0.15,0.89,60.73,0.22,0.15,0.07,...,0.00,0.00,0.07,0.07,0.00,0.07,0.22,0.00,32.27,0.15
Bali,0.94,0.14,0.05,0.09,0.05,0.42,48.31,0.42,0.05,0.14,...,0.00,0.00,0.05,0.00,0.05,0.05,0.09,0.00,42.69,0.00
Barcelona,0.59,0.37,0.00,0.11,0.15,0.33,60.00,0.59,0.15,0.04,...,0.00,0.00,0.07,0.00,0.04,0.00,0.15,0.00,33.65,0.00
Cairo,0.11,0.56,0.00,0.00,0.28,0.28,47.63,0.17,0.17,0.11,...,0.00,0.00,0.00,0.00,0.00,0.00,0.17,0.06,46.05,0.00
Dubai,0.51,0.47,0.00,0.00,0.17,0.47,54.09,0.24,0.07,0.00,...,0.00,0.00,0.07,0.03,0.03,0.07,0.24,0.00,39.88,0.03
Marrakech,0.37,0.41,0.00,0.04,0.29,0.74,55.37,0.21,0.16,0.08,...,0.00,0.00,0.04,0.00,0.12,0.00,0.21,0.00,35.85,0.00
Miami,0.59,0.32,0.00,0.18,0.36,0.50,55.79,0.27,0.09,0.05,...,0.05,0.00,0.09,0.00,0.14,0.09,0.14,0.14,37.21,0.00
Paris,0.51,0.21,0.00,0.08,0.30,0.38,42.91,0.21,0.08,0.08,...,0.00,0.04,0.00,0.00,0.17,0.00,0.04,0.00,51.01,0.00
Rio de Janeiro,0.34,0.11,0.00,0.17,0.11,0.57,41.24,0.68,0.06,0.06,...,0.00,0.00,0.00,0.00,0.00,0.00,0.17,0.00,52.31,0.00


#### Step10D: Investigate "unknown" reviews

In [26]:
# Unknown Reviews By City

unknown_by_city = (
    df[df["Language"] == "unknown"]
    .groupby("City")
    .size()
    .reset_index(name="Unknown_Count")
)

total_by_city = (
    df.groupby("City")
    .size()
    .reset_index(name="Total_Reviews")
)

unknown_summary = total_by_city.merge(
    unknown_by_city,
    on="City",
    how="left"
)

unknown_summary["Unknown_Count"] = (
    unknown_summary["Unknown_Count"].fillna(0).astype(int)
)

unknown_summary["Unknown_Percentage"] = (
    unknown_summary["Unknown_Count"]
    / unknown_summary["Total_Reviews"]
    * 100
).round(2)

display(unknown_summary)

,City,Total_Reviews,Unknown_Count,Unknown_Percentage
0,Agra,1342,433,32.27
1,Bali,2134,911,42.69
2,Barcelona,2725,917,33.65
3,Cairo,1770,815,46.05
4,Dubai,2949,1176,39.88
5,Marrakech,2438,874,35.85
6,Miami,2201,819,37.21
7,Paris,2370,1209,51.01
8,Rio de Janeiro,1753,917,52.31
9,Rome,2410,1210,50.21


In [27]:
unknown_examples = (
    df[df["Language"] == "unknown"]
    .groupby("City", group_keys=False)
    .head(10)
)

display(
    unknown_examples[
        ["City", "Place", "Review_Text", "Language"]
    ]
)

,City,Place,Review_Text,Language
1,Agra,Taj Mahal,Awesome place,unknown
3,Agra,Taj Mahal,Mesmerizing,unknown
6,Agra,Taj Mahal,Jaw dropping wonder,unknown
11,Agra,Taj Mahal,Amazing place,unknown
14,Agra,Taj Mahal,Its got to be done!,unknown
...,...,...,...,...
19694,Rome,Trevi Fountain,Beautiful,unknown
19695,Rome,Trevi Fountain,Trevi,unknown
19696,Rome,Trevi Fountain,April,unknown
19697,Rome,Trevi Fountain,Too many tourists!,unknown


In [28]:
# Unknown Reviews By City
unknown_by_city = (
    df[df["Language"] == "unknown"]
    .groupby("City")
    .size()
    .reset_index(name="Unknown_Count")
)

total_by_city = (
    df.groupby("City")
    .size()
    .reset_index(name="Total_Reviews")
)

unknown_summary = total_by_city.merge(
    unknown_by_city,
    on="City",
    how="left"
)

unknown_summary["Unknown_Count"] = (
    unknown_summary["Unknown_Count"].fillna(0).astype(int)
)

unknown_summary["Unknown_Percentage"] = (
    unknown_summary["Unknown_Count"]
    / unknown_summary["Total_Reviews"]
    * 100
).round(2)

display(unknown_summary)

,City,Total_Reviews,Unknown_Count,Unknown_Percentage
0,Agra,1342,433,32.27
1,Bali,2134,911,42.69
2,Barcelona,2725,917,33.65
3,Cairo,1770,815,46.05
4,Dubai,2949,1176,39.88
5,Marrakech,2438,874,35.85
6,Miami,2201,819,37.21
7,Paris,2370,1209,51.01
8,Rio de Janeiro,1753,917,52.31
9,Rome,2410,1210,50.21


In [29]:
# Check Total Unknown Language Reviews

# Total number of reviews
total_reviews = len(df)

# Number of reviews detected as "unknown"
unknown_reviews = (df["Language"] == "unknown").sum()

# Percentage of unknown reviews
unknown_percentage = (unknown_reviews / total_reviews) * 100

print(f"Total reviews: {total_reviews:,}")
print(f"Unknown reviews: {unknown_reviews:,}")
print(f"Unknown percentage: {unknown_percentage:.2f}%")

print(
    f"\n{unknown_reviews:,} of {total_reviews:,} reviews "
    f"({unknown_percentage:.2f}%) are unknown."
)

Total reviews: 22,092
Unknown reviews: 9,281
Unknown percentage: 42.01%

9,281 of 22,092 reviews (42.01%) are unknown.


In [30]:
 #Analyse Unknown Reviews By Text Length

df["Word_Count"] = (
    df["Review_Text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

unknown_df = df[df["Language"] == "unknown"]

print("Total unknown:", len(unknown_df))

print("\nUnknown reviews by word length:")
print("1-2 words :", (unknown_df["Word_Count"] <= 2).sum())
print("3-5 words :", unknown_df["Word_Count"].between(3, 5).sum())
print("6-10 words:", unknown_df["Word_Count"].between(6, 10).sum())
print("11-20 words:", unknown_df["Word_Count"].between(11, 20).sum())
print(">20 words :", (unknown_df["Word_Count"] > 20).sum())

Total unknown: 9281

Unknown reviews by word length:
1-2 words : 5892
3-5 words : 3386
6-10 words: 3
11-20 words: 0
>20 words : 0


### Step 10E: Create Language Status

In [31]:

def create_language_status(row):
    
    if row["Language"] != "unknown":
        return "detected"
    
    if row["Word_Count"] <= 5:
        return "unknown_short"
    
    return "unknown_other"


df["Language_Status"] = df.apply(
    create_language_status,
    axis=1
)

print(df["Language_Status"].value_counts())

print("\nPercentage:")
print(
    (df["Language_Status"].value_counts(normalize=True) * 100)
    .round(2)
)

Language_Status
detected         12811
unknown_short     9278
unknown_other        3
Name: count, dtype: int64

Percentage:
Language_Status
detected         57.99
unknown_short    42.00
unknown_other     0.01
Name: proportion, dtype: float64


In [32]:
# Check unknown_other reviews
display(
    df[df["Language_Status"] == "unknown_other"][
        [
            "City",
            "Place",
            "Review_Text",
            "Word_Count",
            "Language"
        ]
    ]
)

,City,Place,Review_Text,Word_Count,Language
9038,Marrakech,Jemaa el-Fnaa,Between a 3 and a 4,6,unknown
17957,Rio de Janeiro,Sugarloaf Mountain,A must do in Rio !,6,unknown
18177,Rio de Janeiro,Sugarloaf Mountain,A must to do in Rio,6,unknown


### Step 10F — Secondary Unknown Language Detection

In [33]:
%pip install --upgrade ipykernel ipython jupyter tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
%pip install lingua-language-detector

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
from lingua import LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_all_languages().build()

In [36]:
def detect_language_lingua(text):

    if pd.isna(text):
        return "unknown"

    text = str(text).strip()

    if text == "":
        return "unknown"

    language = detector.detect_language_of(text)

    if language is None:
        return "unknown"

    return language.iso_code_639_1.name.lower()

In [37]:
df["Language_Final"] = df["Language"].copy()

unknown_mask = df["Language"] == "unknown"

print("Unknown before secondary detection:", unknown_mask.sum())

df.loc[unknown_mask, "Language_Final"] = (
    df.loc[unknown_mask, "Review_Text"]
      .apply(detect_language_lingua)
)

print(
    "Unknown after secondary detection:",
    (df["Language_Final"] == "unknown").sum()
)

Unknown before secondary detection: 9281
Unknown after secondary detection: 13


In [38]:
display(
    df[df["Language"] == "unknown"][
        [
            "City",
            "Place",
            "Review_Text",
            "Language",
            "Language_Final"
        ]
    ].head(30)
)

,City,Place,Review_Text,Language,Language_Final
1,Agra,Taj Mahal,Awesome place,unknown,en
3,Agra,Taj Mahal,Mesmerizing,unknown,en
6,Agra,Taj Mahal,Jaw dropping wonder,unknown,en
11,Agra,Taj Mahal,Amazing place,unknown,en
14,Agra,Taj Mahal,Its got to be done!,unknown,tn
21,Agra,Taj Mahal,JUST MAGNIFICENT,unknown,la
23,Agra,Taj Mahal,Finally Taj,unknown,en
32,Agra,Taj Mahal,Excellent Taj,unknown,fr
35,Agra,Taj Mahal,What can i say,unknown,en
45,Agra,Taj Mahal,Speechless,unknown,en


### STEP 10F: Install fastText

In [39]:
%pip install langid

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
import langid
import pandas as pd

def detect_language_langid(text):

    if pd.isna(text):
        return "unknown"

    text = str(text).strip()

    if text == "":
        return "unknown"

    try:
        language, score = langid.classify(text)
        return language

    except Exception:
        return "unknown"

In [41]:
# Example: recheck reviews with 3 words or fewer
short_mask = df["Word_Count"] <= 3

print("Short reviews to recheck:", short_mask.sum())

df.loc[short_mask, "Language_Final"] = (
    df.loc[short_mask, "Review_Text"]
      .apply(detect_language_langid)
)

Short reviews to recheck: 9819


In [42]:
display(
    df[short_mask][
        [
            "Review_Text",
            "Language",
            "Language_Final",
            "Word_Count"
        ]
    ].head(30)
)

,Review_Text,Language,Language_Final,Word_Count
0,An amazing experience,ro,en,3
1,Awesome place,unknown,en,2
3,Mesmerizing,unknown,it,1
6,Jaw dropping wonder,unknown,en,3
8,An ambition acheived,en,gl,3
11,Amazing place,unknown,it,2
17,Fantastic experience,es,en,2
19,Better second-time visit!,it,en,3
21,JUST MAGNIFICENT,unknown,en,2
23,Finally Taj,unknown,en,2


### Step 10F: Final Language Quality Summary

In [43]:

total = len(df)

# Final language distribution
language_counts = df["Language_Final"].value_counts(dropna=False)

summary = pd.DataFrame({
    "Count": language_counts,
    "Percentage": (
        df["Language_Final"]
        .value_counts(normalize=True, dropna=False) * 100
    ).round(2)
})

print("=" * 60)
print("FINAL LANGUAGE QUALITY SUMMARY")
print("=" * 60)

print("\nFinal detected languages:")
display(summary)

# English / Non-English / Unknown summary
english_count = (df["Language_Final"] == "en").sum()
unknown_count = (df["Language_Final"] == "unknown").sum()

non_english_count = (
    (df["Language_Final"] != "en") &
    (df["Language_Final"] != "unknown") &
    (df["Language_Final"].notna())
).sum()

status_summary = pd.DataFrame({
    "Count": [
        english_count,
        non_english_count,
        unknown_count
    ],
    "Percentage": [
        round(english_count / total * 100, 2),
        round(non_english_count / total * 100, 2),
        round(unknown_count / total * 100, 2)
    ]
},
index=[
    "English",
    "Non-English",
    "Unknown"
])

print("\nLanguage Status:")
display(status_summary)

print("\nTop 15 detected languages:")
print(
    df["Language_Final"]
    .value_counts()
    .head(15)
)

print("\nDataset size:", f"{total:,}")

print(
    "Unknown reviews remaining:",
    f"{unknown_count:,}"
)

FINAL LANGUAGE QUALITY SUMMARY

Final detected languages:


,Count,Percentage
Language_Final,,
en,18259,82.65
fr,541,2.45
it,519,2.35
de,497,2.25
nl,470,2.13
...,...,...
vo,1,0.00
br,1,0.00
ar,1,0.00



Language Status:


,Count,Percentage
English,18259,82.65
Non-English,3833,17.35
Unknown,0,0.00



Top 15 detected languages:
Language_Final
en    18259
fr      541
it      519
de      497
nl      470
es      367
ro      157
fi      147
et      139
da       99
af       89
id       89
pt       76
pl       72
ca       60
Name: count, dtype: int64

Dataset size: 22,092
Unknown reviews remaining: 0


Step 11 — Add a unique Review ID

In [44]:
df = df.reset_index(drop=True)

df["Review_ID"] = [
    f"R{i:06d}"
    for i in range(1, len(df) + 1)
]

df[
    [
        "Review_ID",
        "City",
        "Place",
        "Review_Date",
        "Review_Text"
    ]
].head(10)

,Review_ID,City,Place,Review_Date,Review_Text
0,R000001,Agra,Taj Mahal,2019-02-17,An amazing experience
1,R000002,Agra,Taj Mahal,2019-02-17,Awesome place
2,R000003,Agra,Taj Mahal,2019-02-17,Definitely a wonder of the world
3,R000004,Agra,Taj Mahal,2019-02-16,Mesmerizing
4,R000005,Agra,Taj Mahal,2019-02-16,A place really called as wonder
5,R000006,Agra,Taj Mahal,2019-02-16,It's was a magnificent experience to be seeing...
6,R000007,Agra,Taj Mahal,2019-02-16,Jaw dropping wonder
7,R000008,Agra,Taj Mahal,2019-02-16,No wonder it's a wonder!
8,R000009,Agra,Taj Mahal,2019-02-16,An ambition acheived
9,R000010,Agra,Taj Mahal,2019-02-16,This is definitely one of the most amazing pla...


#### Step 12A: Examine Complete Language Distribution

In [45]:


language_summary = (
    df["Language_Final"]
    .value_counts(dropna=False)
    .rename_axis("Language")
    .reset_index(name="Review_Count")
)

language_summary["Percentage"] = (
    language_summary["Review_Count"]
    / len(df) * 100
).round(2)

language_summary

,Language,Review_Count,Percentage
0,en,18259,82.65
1,fr,541,2.45
2,it,519,2.35
3,de,497,2.25
4,nl,470,2.13
...,...,...,...
59,vo,1,0.00
60,br,1,0.00
61,ar,1,0.00
62,el,1,0.00


#### Step 12b — Inspect the non-English reviews

In [46]:

non_english = df[
    ~df["Language_Final"].isin(["en", "unknown"])
]

print("Final non-English reviews:", len(non_english))

non_english[
    [
        "Review_ID",
        "City",
        "Place",
        "Language_Final",
        "Word_Count",
        "Review_Text"
    ]
].sample(
    min(30, len(non_english)),
    random_state=42
)

Final non-English reviews: 3833


,Review_ID,City,Place,Language_Final,Word_Count,Review_Text
14272,R014273,Cairo,Egyptian Museum,fr,5,Excellent collection in old museum
14518,R014519,Cairo,Egyptian Museum,fr,4,Museum of Egyptian Antiquities
20393,R020394,Rome,Colosseum,it,1,Amazing
16423,R016424,Paris,Louvre Museum,nl,3,So worth it
17288,R017289,Paris,Arc de Triomphe,et,2,Marvelous monument
7655,R007656,Marrakech,Jardin Majorelle,da,3,Check your tickets
4698,R004699,Dubai,Dubai Mall,es,2,Fantasy land?
19685,R019686,Rome,Trevi Fountain,fr,2,Marvelous design
13082,R013083,Bali,Uluwatu Temple,it,2,Amazing Views
11097,R011098,Miami,Miami Beach Boardwalk,de,3,Morning Beach Stroll


### STEP 12c — Language distribution for longer reviews


In [47]:

long_reviews = df[df["Word_Count"] >= 10].copy()

print("Total dataset:", len(df))
print("Reviews with >=10 words:", len(long_reviews))

print(
    "Percentage with >=10 words:",
    round(len(long_reviews) / len(df) * 100, 2),
    "%"
)

print("\nLanguage distribution for reviews >=10 words:")

long_language_summary = (
    long_reviews["Language_Final"]
    .value_counts()
    .rename_axis("Language")
    .reset_index(name="Count")
)

long_language_summary["Percentage"] = (
    long_language_summary["Count"]
    / len(long_reviews) * 100
).round(2)

display(long_language_summary.head(20))

Total dataset: 22092
Reviews with >=10 words: 2760
Percentage with >=10 words: 12.49 %

Language distribution for reviews >=10 words:


,Language,Count,Percentage
0,en,2745,99.46
1,af,7,0.25
2,fr,3,0.11
3,de,2,0.07
4,sv,1,0.04
5,cy,1,0.04
6,pt,1,0.04


### 12D Inspect non-English reviews where there is enough text


In [48]:
# Inspect non-English reviews where there is enough text

long_non_english = df[
    (df["Word_Count"] >= 10) &
    (~df["Language_Final"].isin(["en", "unknown"]))
].copy()

print(
    "Non-English detected reviews with >=10 words:",
    len(long_non_english)
)

display(
    long_non_english[
        [
            "Review_ID",
            "City",
            "Place",
            "Language_Final",
            "Word_Count",
            "Review_Text"
        ]
    ].sample(
        min(30, len(long_non_english)),
        random_state=42
    )
)

Non-English detected reviews with >=10 words: 15


,Review_ID,City,Place,Language_Final,Word_Count,Review_Text
9842,R009843,Miami,Wynwood Walls,cy,12,You e not done Miami if you e not done Wynwood...
18344,R018345,Rio de Janeiro,Sugarloaf Mountain,af,11,Indeed Beautiful over looking views on top of ...
811,R000812,Agra,Agra Fort,de,10,RED FORT - FORTRESS WITH VIEW OF THE TAJ MAHAL
19469,R019470,Rio de Janeiro,Santuário Cristo Redentor,pt,12,A must to see if you go to Rio de Janeiro !
6715,R006716,Dubai,Dubai Frame,af,13,"Must visit in Dubai! ????? ?????????? ?????, ?..."
9451,R009452,Marrakech,Jemaa el-Fnaa,sv,10,Fresh juice stalls in jemaa el fna (avoid stal...
1961,R001962,Barcelona,Park Güell,af,10,A must-visit. Feels like transported into Hans...
868,R000869,Agra,Agra Fort,af,10,MUST visit in Agra. Make sure to get a guide
20446,R020447,Rome,Colosseum,fr,10,Do not use Tiqets or Omnia if you visit Colosseum
6705,R006706,Dubai,Dubai Frame,de,10,"ONE OF DUBAI'S NEWEST ATTRACTIONS, AMAZING VIE..."


Step 13A — Install translator

In [49]:
%pip install deep-translator

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Step 13B — Import and identify reviews requiring translation

In [50]:
from deep_translator import GoogleTranslator
import pandas as pd
import time

# Reviews requiring translation

translation_mask = (
    df["Language_Final"].notna()
    & (~df["Language_Final"].isin(["en", "unknown"]))
)

print("Total reviews:", len(df))
print("English reviews:", (df["Language_Final"] == "en").sum())
print("Reviews requiring translation:", translation_mask.sum())
print("Unknown reviews:", (df["Language_Final"] == "unknown").sum())

Total reviews: 22092
English reviews: 18259
Reviews requiring translation: 3833
Unknown reviews: 0


Step 13C — Create the translation columns

In [51]:
# Preserve original text
df["Translated_Text"] = df["Review_Text"].copy()

# Translation status
df["Translation_Status"] = "Not Required"

df.loc[
    translation_mask,
    "Translation_Status"
] = "Pending"

df.loc[
    df["Language_Final"] == "unknown",
    "Translation_Status"
] = "Unknown - Original Retained"

Step 13D — Create translation function

In [52]:
def translate_to_english(text):

    if pd.isna(text):
        return text

    text = str(text).strip()

    if text == "":
        return text

    try:
        translated = GoogleTranslator(
            source="auto",
            target="en"
        ).translate(text)

        return translated

    except Exception as e:
        print("Translation error:", e)

        # IMPORTANT:
        # Keep original instead of losing data
        return text

Step 13E — Test it before translating everything

In [53]:
test_translation = df[
    translation_mask
].head(10).copy()

test_translation["Test_Translation"] = (
    test_translation["Review_Text"]
    .apply(translate_to_english)
)

display(
    test_translation[
        [
            "Language_Final",
            "Review_Text",
            "Test_Translation"
        ]
    ]
)

,Language_Final,Review_Text,Test_Translation
3,it,Mesmerizing,Mesmerizing
7,nl,No wonder it's a wonder!,No wonder it's a wonder!
8,gl,An ambition acheived,An ambition acheived
11,it,Amazing place,Amazing place
14,tn,Its got to be done!,Its got to be done!
32,de,Excellent Taj,Excellent Taj
33,it,More magical in person,More magical in person
45,de,Speechless,Speechless
48,de,Unbelievable,Unbelievable
50,es,Spectacular!,Spectacular!


### Step 13F — translate all candidate non-English reviews.

In [54]:
indices_to_translate = df[
    translation_mask
].index.tolist()

print("Starting translations:", len(indices_to_translate))

for count, idx in enumerate(indices_to_translate, start=1):

    original_text = df.at[idx, "Review_Text"]

    try:
        translated_text = translate_to_english(original_text)

        df.at[idx, "Translated_Text"] = translated_text

        # Translation process completed successfully
        df.at[idx, "Translation_Status"] = "Processed"

    except Exception as e:

        # Preserve original review if translation actually fails
        df.at[idx, "Translated_Text"] = original_text
        df.at[idx, "Translation_Status"] = "Translation Failed - Original Retained"

        print(
            f"Error at Review_ID {df.at[idx, 'Review_ID']}: {e}"
        )

    # Save checkpoint every 100 reviews
    if count % 100 == 0:

        df.to_csv(
            "translation_checkpoint.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"{count} / {len(indices_to_translate)} completed"
        )

    time.sleep(0.1)

Starting translations: 3833
100 / 3833 completed
200 / 3833 completed
300 / 3833 completed
400 / 3833 completed
500 / 3833 completed
600 / 3833 completed
700 / 3833 completed
800 / 3833 completed
900 / 3833 completed
1000 / 3833 completed
1100 / 3833 completed
1200 / 3833 completed
1300 / 3833 completed
1400 / 3833 completed
1500 / 3833 completed
1600 / 3833 completed
1700 / 3833 completed
1800 / 3833 completed
1900 / 3833 completed
2000 / 3833 completed
2100 / 3833 completed
2200 / 3833 completed
2300 / 3833 completed
2400 / 3833 completed
2500 / 3833 completed
2600 / 3833 completed
2700 / 3833 completed
2800 / 3833 completed
2900 / 3833 completed
3000 / 3833 completed
3100 / 3833 completed
3200 / 3833 completed
3300 / 3833 completed
3400 / 3833 completed
3500 / 3833 completed
3600 / 3833 completed
3700 / 3833 completed
3800 / 3833 completed


In [55]:
# ==========================================
# STEP 13G — TRANSLATION SUMMARY
# ==========================================

translation_summary = (
    df["Translation_Status"]
    .value_counts(dropna=False)
    .rename_axis("Translation_Status")
    .reset_index(name="Count")
)

translation_summary["Percentage"] = (
    translation_summary["Count"]
    / len(df) * 100
).round(2)

display(translation_summary)

,Translation_Status,Count,Percentage
0,Not Required,18259,82.65
1,Processed,3833,17.35


In [56]:
# ==========================================
# STEP 13H — CHECK RANDOM TRANSLATIONS
# ==========================================

processed_reviews = df[
    df["Translation_Status"] == "Processed"
]

display(
    processed_reviews[
        [
            "Review_ID",
            "Language_Final",
            "Review_Text",
            "Translated_Text"
        ]
    ].sample(
        min(30, len(processed_reviews)),
        random_state=42
    )
)

,Review_ID,Language_Final,Review_Text,Translated_Text
14272,R014273,fr,Excellent collection in old museum,Excellent collection in old museum
14518,R014519,fr,Museum of Egyptian Antiquities,Museum of Egyptian Antiquities
20393,R020394,it,Amazing,Amazing
16423,R016424,nl,So worth it,So worth it
17288,R017289,et,Marvelous monument,Marvelous monument
7655,R007656,da,Check your tickets,Check your tickets
4698,R004699,es,Fantasy land?,Fantasy land?
19685,R019686,fr,Marvelous design,Marvelous design
13082,R013083,it,Amazing Views,Amazing Views
11097,R011098,de,Morning Beach Stroll,Morning Beach Stroll


In [57]:
# ==========================================
# STEP 13H — TRANSLATION QUALITY CHECK
# ==========================================

processed_reviews = df[
    df["Translation_Status"] == "Processed"
]

print("Processed reviews:", len(processed_reviews))

display(
    processed_reviews[
        [
            "Review_ID",
            "Language_Final",
            "Review_Text",
            "Translated_Text"
        ]
    ].sample(
        min(30, len(processed_reviews)),
        random_state=42
    )
)

Processed reviews: 3833


,Review_ID,Language_Final,Review_Text,Translated_Text
14272,R014273,fr,Excellent collection in old museum,Excellent collection in old museum
14518,R014519,fr,Museum of Egyptian Antiquities,Museum of Egyptian Antiquities
20393,R020394,it,Amazing,Amazing
16423,R016424,nl,So worth it,So worth it
17288,R017289,et,Marvelous monument,Marvelous monument
7655,R007656,da,Check your tickets,Check your tickets
4698,R004699,es,Fantasy land?,Fantasy land?
19685,R019686,fr,Marvelous design,Marvelous design
13082,R013083,it,Amazing Views,Amazing Views
11097,R011098,de,Morning Beach Stroll,Morning Beach Stroll


In [58]:
# ==========================================
# STEP 13I — DATA LOSS CHECK
# ==========================================

print("Total reviews:", f"{len(df):,}")
print("Unique Review IDs:", f"{df['Review_ID'].nunique():,}")

print(
    "Missing original text:",
    df["Review_Text"].isna().sum()
)

print(
    "Missing translated text:",
    df["Translated_Text"].isna().sum()
)

print(
    "Empty translated text:",
    (
        df["Translated_Text"]
        .fillna("")
        .str.strip()
        == ""
    ).sum()
)

Total reviews: 22,092
Unique Review IDs: 22,092
Missing original text: 0
Missing translated text: 7
Empty translated text: 7


In [59]:
# ==========================================
# STEP 13J — CREATE MODEL TEXT
# ==========================================

df["Model_Text"] = df["Translated_Text"].copy()

print("Model_Text created successfully.")

print("Missing Model_Text:", df["Model_Text"].isna().sum())
print(
    "Empty Model_Text:",
    (df["Model_Text"].fillna("").str.strip() == "").sum()
)

Model_Text created successfully.
Missing Model_Text: 7
Empty Model_Text: 7


### Step 13K — Final Data Quality Audit

In [60]:
print("=" * 60)
print("FINAL DATA QUALITY AUDIT")
print("=" * 60)

# 1. Dataset size
print("\n1. DATASET SIZE")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# 2. Duplicates
print("\n2. DUPLICATES")
print("Exact duplicate rows:", df.duplicated().sum())

# 3. Review ID
print("\n3. REVIEW ID")
print("Unique Review_ID:", df["Review_ID"].nunique())
print("Duplicate Review_ID:", df["Review_ID"].duplicated().sum())

# 4. Date range
print("\n4. DATE RANGE")
print("Earliest:", df["Review_Date"].min())
print("Latest:", df["Review_Date"].max())

# 5. Ratings
print("\n5. RATINGS")
print(df["Rating"].value_counts().sort_index())

print(
    "Invalid ratings:",
    (~df["Rating"].between(1, 5)).sum()
)

# 6. Missing values
print("\n6. MISSING VALUES")

missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

print(
    missing[
        missing["Missing_Count"] > 0
    ]
)

# 7. Cities
print("\n7. CITIES")
print("Number of cities:", df["City"].nunique())

print(
    df["City"]
    .value_counts()
    .sort_index()
)

# 8. Attractions
print("\n8. ATTRACTIONS")
print("Number of attractions:", df["Place"].nunique())

# 9. Reviews per year
print("\n9. REVIEWS PER YEAR")

print(
    df["Year"]
    .value_counts()
    .sort_index()
)

# 10. Short reviews
print("\n10. SHORT REVIEWS")

print(
    "Reviews <=5 words:",
    (df["Word_Count"] <= 5).sum()
)

print(
    "Percentage:",
    round(
        (df["Word_Count"] <= 5).mean() * 100,
        2
    ),
    "%"
)

FINAL DATA QUALITY AUDIT

1. DATASET SIZE
Rows: 22092
Columns: 21

2. DUPLICATES
Exact duplicate rows: 0

3. REVIEW ID
Unique Review_ID: 22092
Duplicate Review_ID: 0

4. DATE RANGE
Earliest: 2019-01-03 00:00:00
Latest: 2024-12-31 00:00:00

5. RATINGS
Rating
1     1046
2      854
3     2176
4     4927
5    13089
Name: count, dtype: int64
Invalid ratings: 0

6. MISSING VALUES
                 Missing_Count  Missing_Percentage
Model_Text                   7                0.03
Translated_Text              7                0.03

7. CITIES
Number of cities: 10
City
Agra              1342
Bali              2134
Barcelona         2725
Cairo             1770
Dubai             2949
Marrakech         2438
Miami             2201
Paris             2370
Rio de Janeiro    1753
Rome              2410
Name: count, dtype: int64

8. ATTRACTIONS
Number of attractions: 48

9. REVIEWS PER YEAR
Year
2019    4717
2020    4022
2021    2043
2022    3557
2023    3754
2024    3999
Name: count, dtype: int64

10. 

#### Step 14: 

#### Step 14A:  City × year coverage

In [61]:
city_year = pd.crosstab(
    df["City"],
    df["Year"]
)

display(city_year)

Year,2019,2020,2021,2022,2023,2024
City,,,,,,
Agra,435,251,85,179,185,207
Bali,734,323,42,320,351,364
Barcelona,500,587,154,580,394,510
Cairo,424,252,218,233,284,359
Dubai,496,482,471,490,531,479
Marrakech,500,375,101,429,482,551
Miami,577,385,310,304,333,292
Paris,312,601,221,424,386,426
Rio de Janeiro,382,322,151,174,357,367


### Step 14B City × month coverage

In [62]:
monthly_counts = (
    df.groupby(
        ["City", "Year", "Month"]
    )
    .size()
    .reset_index(name="Review_Count")
)

display(monthly_counts.head(20))

print("Number of observed city-months:",
      len(monthly_counts))

print("Minimum reviews/month:",
      monthly_counts["Review_Count"].min())

print("Median reviews/month:",
      monthly_counts["Review_Count"].median())

print("Mean reviews/month:",
      round(monthly_counts["Review_Count"].mean(), 2))

print("Maximum reviews/month:",
      monthly_counts["Review_Count"].max())

,City,Year,Month,Review_Count
0,Agra,2019,1,10
1,Agra,2019,2,146
2,Agra,2019,3,67
3,Agra,2019,4,22
4,Agra,2019,5,38
5,Agra,2019,6,41
6,Agra,2019,7,26
7,Agra,2019,8,22
8,Agra,2019,9,20
9,Agra,2019,10,19


Number of observed city-months: 675
Minimum reviews/month: 1
Median reviews/month: 25.0
Mean reviews/month: 32.73
Maximum reviews/month: 213


### Step 14C — Find Missing City-Months

In [64]:
all_months = pd.MultiIndex.from_product(
    [
        sorted(df["City"].unique()),
        range(2019, 2025),
        range(1, 13)
    ],
    names=["City", "Year", "Month"]
)

monthly_complete = (
    monthly_counts
    .set_index(["City", "Year", "Month"])
    .reindex(all_months, fill_value=0)
    .reset_index()
)

missing_months = monthly_complete[
    monthly_complete["Review_Count"] == 0
].copy()

print("Expected city-months:", len(monthly_complete))

print(
    "Observed city-months:",
    (monthly_complete["Review_Count"] > 0).sum()
)

print("Missing city-months:", len(missing_months))

print(
    "Coverage:",
    round(
        (monthly_complete["Review_Count"] > 0).mean() * 100,
        2
    ),
    "%"
)

display(missing_months)

Expected city-months: 720
Observed city-months: 675
Missing city-months: 45
Coverage: 93.75 %


,City,Year,Month,Review_Count
17,Agra,2020,6,0
23,Agra,2020,12,0
29,Agra,2021,6,0
72,Bali,2019,1,0
102,Bali,2021,7,0
152,Barcelona,2019,9,0
153,Barcelona,2019,10,0
154,Barcelona,2019,11,0
155,Barcelona,2019,12,0
170,Barcelona,2021,3,0


In [65]:
missing_by_city = (
    missing_months
    .groupby("City")
    .size()
    .reset_index(name="Missing_Months")
    .sort_values("Missing_Months", ascending=False)
)

display(missing_by_city)

,City,Missing_Months
6,Paris,9
8,Rome,8
5,Marrakech,7
2,Barcelona,6
4,Dubai,5
0,Agra,3
3,Cairo,3
1,Bali,2
7,Rio de Janeiro,2


Step 14D — Investigate Low-Sample Months


In [66]:
thresholds = [1, 5, 10, 20, 30]

for threshold in thresholds:

    count = (
        monthly_complete["Review_Count"] < threshold
    ).sum()

    percentage = (
        count / len(monthly_complete) * 100
    )

    print(
        f"< {threshold:2d} reviews : "
        f"{count:3d} city-months "
        f"({percentage:.2f}%)"
    )

<  1 reviews :  45 city-months (6.25%)
<  5 reviews :  93 city-months (12.92%)
< 10 reviews : 169 city-months (23.47%)
< 20 reviews : 312 city-months (43.33%)
< 30 reviews : 448 city-months (62.22%)


In [67]:
observed_months = monthly_complete[
    monthly_complete["Review_Count"] > 0
].copy()

print("ONLY MONTHS THAT CONTAIN REVIEWS\n")

for threshold in [5, 10, 20, 30]:

    count = (
        observed_months["Review_Count"] < threshold
    ).sum()

    percentage = (
        count / len(observed_months) * 100
    )

    print(
        f"1-{threshold-1} reviews: "
        f"{count} months "
        f"({percentage:.2f}%)"
    )

ONLY MONTHS THAT CONTAIN REVIEWS

1-4 reviews: 48 months (7.11%)
1-9 reviews: 124 months (18.37%)
1-19 reviews: 267 months (39.56%)
1-29 reviews: 403 months (59.70%)


### Step 14E — Monthly Coverage by Year

In [68]:
year_quality = (
    monthly_complete
    .groupby("Year")
    .agg(
        Total_City_Months=("Review_Count", "size"),

        Months_With_Data=(
            "Review_Count",
            lambda x: (x > 0).sum()
        ),

        Missing_Months=(
            "Review_Count",
            lambda x: (x == 0).sum()
        ),

        Low_Under_10=(
            "Review_Count",
            lambda x: ((x > 0) & (x < 10)).sum()
        ),

        Median_Reviews=("Review_Count", "median"),

        Total_Reviews=("Review_Count", "sum")
    )
)

display(year_quality)

,Total_City_Months,Months_With_Data,Missing_Months,Low_Under_10,Median_Reviews,Total_Reviews
Year,,,,,,
2019,120,86,34,9,20.5,4717
2020,120,116,4,39,15.0,4022
2021,120,115,5,41,13.0,2043
2022,120,120,0,20,25.0,3557
2023,120,119,1,9,28.0,3754
2024,120,119,1,6,29.0,3999


In [69]:
missing_2019 = monthly_complete[
    (monthly_complete["Year"] == 2019) &
    (monthly_complete["Review_Count"] == 0)
]

display(
    missing_2019[
        ["City", "Year", "Month", "Review_Count"]
    ].sort_values(["City", "Month"])
)

,City,Year,Month,Review_Count
72,Bali,2019,1,0
152,Barcelona,2019,9,0
153,Barcelona,2019,10,0
154,Barcelona,2019,11,0
155,Barcelona,2019,12,0
218,Cairo,2019,3,0
224,Cairo,2019,9,0
288,Dubai,2019,1,0
296,Dubai,2019,9,0
297,Dubai,2019,10,0


In [70]:
missing_2019_summary = (
    missing_2019
    .groupby("City")
    .size()
    .reset_index(name="Missing_Months_2019")
    .sort_values(
        "Missing_Months_2019",
        ascending=False
    )
)

display(missing_2019_summary)

,City,Missing_Months_2019
5,Paris,9
7,Rome,7
4,Marrakech,5
1,Barcelona,4
3,Dubai,4
2,Cairo,2
6,Rio de Janeiro,2
0,Bali,1


In [71]:
# ==========================================
# STEP 14G — COVERAGE SUMMARY
# ==========================================

coverage_summary = (
    monthly_complete
    .groupby("City")
    .agg(
        Observed_Months=(
            "Review_Count",
            lambda x: (x > 0).sum()
        ),
        Missing_Months=(
            "Review_Count",
            lambda x: (x == 0).sum()
        ),
        Total_Reviews=(
            "Review_Count",
            "sum"
        )
    )
)

coverage_summary["Coverage_Percentage"] = (
    coverage_summary["Observed_Months"]
    / 72 * 100
).round(2)

display(coverage_summary)

,Observed_Months,Missing_Months,Total_Reviews,Coverage_Percentage
City,,,,
Agra,69,3,1342,95.83
Bali,70,2,2134,97.22
Barcelona,66,6,2725,91.67
Cairo,69,3,1770,95.83
Dubai,67,5,2949,93.06
Marrakech,65,7,2438,90.28
Miami,72,0,2201,100.00
Paris,63,9,2370,87.50
Rio de Janeiro,70,2,1753,97.22


In [72]:
# ==========================================
# STEP 15 — SAVE FINAL PREPROCESSED DATASET
# ==========================================

output_file = "Tourism_Reviews_Preprocessed.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset saved successfully.")
print("File:", output_file)
print("Total reviews:", len(df))
print("Cities:", df["City"].nunique())
print("Attractions:", df["Place"].nunique())

Dataset saved successfully.
File: Tourism_Reviews_Preprocessed.csv
Total reviews: 22092
Cities: 10
Attractions: 48


## Preprocessing Summary

The preprocessing stage prepared the tourism review dataset for the subsequent sentiment, emotion, temporal drift, and event-mapping analyses.

### Final Dataset
- **Total reviews:** 22,092
- **Cities:** 10
- **Tourist attractions:** 48
- **Study period:** 2019–2024
- **Final output:** `Tourism_Reviews_Preprocessed.csv`

### Preprocessing Performed
The main preprocessing activities included:

1. Combined the individual attraction-level review datasets into a single dataset.
2. Standardized the dataset structure and reviewed missing and duplicate records.
3. Created a unique `Review_ID` for traceability of each review.
4. Processed review dates and extracted temporal variables such as year and month.
5. Standardized and validated review ratings.
6. Calculated review-level characteristics such as word count.
7. Performed language identification and additional checks for uncertain and short reviews.
8. Preserved the original review text while translating/normalizing non-English reviews into English.
9. Created `Model_Text` as the standardized textual input for subsequent NLP analysis.
10. Performed final data-quality and temporal-coverage checks.

### Temporal Coverage
Monthly coverage was evaluated across the complete 2019–2024 study period. Coverage varied by city from **87.50% to 100%**, with Miami having complete monthly coverage. Missing city-month observations were retained as missing rather than artificially imputed.

### Final Outcome
The resulting dataset preserves the original review information while providing standardized text and temporal variables suitable for downstream analysis. No synthetic reviews were introduced to compensate for missing periods.

The next stage applies emotion classification to `Model_Text`, followed by monthly aggregation, emotional-drift measurement, change/spike detection, and real-world event mapping.